# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library with a focus on referencing all entities by their `@id` fields as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.datePublished}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s, and all fields in the dataset.

In [ ]:
# List all record set @ids and their fields
record_sets = dataset.metadata.recordSet
print("Available Record Sets and their Fields:")

overview = []
for rs in record_sets:
    rs_id = rs['@id']
    rs_name = rs.get('name', '--')
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nRecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    print(f"  Fields:")
    for field in fields:
        fid = field['@id']
        fname = field.get('name', '--')
        ftype = field.get('dataType', '--')
        print(f"    - @id: {fid} | name: {fname} | type: {ftype}")
    overview.append({'record_set_id': rs_id, 'fields': fields})

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we will demonstrate using the principal data record set which likely holds the main survey data. Replace the `record_set_id` and field `@id`s with those found in the previous cell if you want to explore other record sets.

In [ ]:
# Find likely main record set (survey) from the overview above.
# For this dataset, let's use the first listed RecordSet.
record_set_id = record_sets[0]['@id']

# You may list all record set @ids as follows:
print("All available record set @ids:")
for rs in record_sets:
    print(rs['@id'])

print(f"\nExtracting records from record set: {record_set_id}")

# Load data as DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print(f"Columns in {record_set_id}:")
print(df.columns.tolist())

df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on specific numeric field and threshold
- Normalizing values
- Grouping by a key attribute

> Ensure your field references use the field `@id`. If the field @ids are different from column names, map accordingly.

In [ ]:
# Select a numeric field for analysis; adjust based on actual column names in df
# Let's assume the GAD-7 score field with the @id 'gad7_score' exists (replace with actual @id if different)
numeric_field = 'gad7_score'  # Replace with numeric field @id from your schema/overview if needed

if numeric_field not in df.columns:
    print(f"Column '{numeric_field}' not found. Sample columns are: {df.columns.tolist()}")

# Example threshold for demonstration
threshold = 5
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by demographic field, e.g., 'gender' if present (replace with correct @id)
    group_field = 'gender'  # Replace with field @id as needed
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        print(grouped_df)
else:
    print("No relevant numeric field available for EDA.")

## 5. Visualization
Visualize distributions or field relationships with matplotlib/seaborn.

Below, we demonstrate a histogram and a boxplot on the GAD-7 score, grouping by gender if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field} (GAD-7 Score)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (e.g. gender)
    if group_field in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded the Croissant-schematized FAIR² Mental Health Survey dataset from Kilifi County, explored the metadata and structure by referencing all entities by their `@id`, extracted and analyzed records from the principal record set, and visualized key numeric fields. This demonstrates data preparation for further statistical or ML-ready analysis. For other advanced uses, repeat the workflow with different record sets or fields as needed.